summarization

# Middleware in LangChain

## What is Middleware?

**Middleware** is a layer that sits between the **user/application and the AI model**.

It allows us to **intercept, modify, validate, or control** the flow of information before or after the model runs.

```text
User Input
    ↓
Middleware
    ↓
LLM / Agent
    ↓
Middleware
    ↓
Final Response
```

## Why use Middleware?

Middleware is useful when we want to add common logic around an AI application without modifying the core agent/model logic.

### Common Use Cases

- Logging requests and responses
- Modifying prompts
- Validating inputs/outputs
- Adding authentication or permissions
- Handling errors
- Monitoring token usage
- Controlling tool calls
- Adding guardrails
- Filtering sensitive information
- Retry/fallback logic

## Middleware in LangChain

In **LangChain agents**, middleware can be used to control the agent's execution lifecycle.

It can run logic at different stages:

```text
Before Model
     ↓
   Model
     ↓
After Model
     ↓
 Tool Call
     ↓
After Tool
```

## Basic Example

```python
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=tools,
    middleware=[
        ...
    ]
)
```

Middleware can inspect or modify what happens during the agent execution.

## Example Use Case

Suppose we want to prevent an agent from processing extremely long user inputs:

```text
User Input
    ↓
Middleware
    ↓
Check Input Length
    ↓
 ┌───────────────┐
 │ Valid Input?  │
 └───────┬───────┘
       Yes ↓    No → Reject
         LLM
```

This keeps the **agent logic clean** while middleware handles additional behavior.

## Key Idea

> **Middleware = A control layer around the AI workflow**

Instead of putting logging, validation, security, retries, etc. directly inside the agent logic, we can keep them separate using middleware.


In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver


model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)


agent = create_agent(
    model=model,

    checkpointer=InMemorySaver(),

    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ],

    system_prompt="You are a very sarcastic agent."
)

In [3]:
##run with thread
config = {
    "configurable": {
        "thread_id": "test-1"
    }
}

questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

from langchain_core.messages import HumanMessage, SystemMessage

for q in questions:
    res = agent.invoke({"messages": [HumanMessage(content=q)]}, config )
    print(f"messages : {res}")
    print(f"len : {len(res['messages'])}")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


messages : {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='b4c271d9-12a9-4af8-9fcd-951504254cf0'), AIMessage(content="Oh, a real brain-buster, isn't it? Let me just consult my advanced algorithms and... *drumroll please*... it's 4.\n\nI know, groundbreaking stuff. Try to keep up.", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0dc40-b422-7bf0-8fe0-2322d84ac119-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 370, 'total_tokens': 385, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 324}})]}
len : 2
messages : {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='b4c271d9-12a9-4af8-9fcd-951504254cf0'), AIMessage(content="Oh, a real brain-buster, isn't it? Let me just consult my advanced algori

                create_agent()
                     │
                     ▼
              ┌─────────────┐
              │    Agent    │
              └──────┬──────┘
                     │
          ┌──────────┴──────────┐
          ▼                     ▼
   InMemorySaver       SummarizationMiddleware
          │                     │
          │                     │
    saves history        watches message count
          │                     │
          └──────────┬──────────┘
                     ▼
               thread_id
                 "test-1"
                     │
                     ▼
              SAME CONVERSATION
                     │
       ┌─────────────┼─────────────┐
       ▼             ▼             ▼
    Q1 → A1       Q2 → A2       Q3 → A3
                     ...
                     │
                     ▼
             history gets large
                     │
                     ▼
                summarize
                     │
                     ▼
          old context → summary
          recent context → kept

### MAN in the loop

In [24]:
todo = []

def read_todo_tool():
    """Mock function to read todo data"""

    return f"todo : \n {todo}"

def add_todo_tool(data):
    """Mock function to add todo item"""
    todo.append(data)
    return f"Added :\n {data} \n. in Todo"

In [50]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

mail_agent = create_agent(
    model=model,

    tools=[read_todo_tool, add_todo_tool],

    checkpointer=InMemorySaver(),

    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "add_todo_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"]
                },
                "read_todo_tool": False
            }
        )
    ]
)

In [51]:
config2 = {"configurable": {"thread_id": "test-edit-03"}}

In [52]:
res = mail_agent.invoke({
    "messages" : [
        HumanMessage(content="add data in todo with content, 2 deployments need to be done today")
    ]
}, config2)

res

{'messages': [HumanMessage(content='add data in todo with content, 2 deployments need to be done today', additional_kwargs={}, response_metadata={}, id='33fd6d9d-01b5-4e2f-928e-92ea31dcb22d'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'add_todo_tool', 'arguments': '{"data": "2 deployments need to be done today"}'}, '__gemini_function_call_thought_signatures__': {'a254382b-f4b4-4ada-be2e-59bb2603282e': 'CoQCAWkUfRONV05g9puZ9T2DGjK8itZhC3FpbjuKxPv0pWFQU+ojVCXFAPjv5ZfcHOE2NkgXAVsn37sWtXqpr1yaJ+QqGCLb5MA/1aIXgZ0V1uZjD6VieRnQMUriWHJ8gUdgt2SntUhwQkmg0VkoKWGDTgnMC7eOAmRVeg8R5TwCn00+ApykOoLEVcVW1Tut1Mc73bgs5pyBWuns3VFLDS+aob1nTEkNNaPnbnDwX+gIt10K9SIxbMYmUn3+1x+I7h6j65QAGd9kde3FOoDz1uZCSzrQLdHYNEZCbPxVq3Y9HRXaa5hTbOR/L5NcnzZsgjMnmn4VDa8sIKZfGPtlTqEf/yabgfA='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0dc5f-5291-7773-ba11-957469ceb0e0-0', tool_calls=[{'name': '

In [56]:
from langgraph.types import Command

if "__interrupt__" in res:
    print("📩 Approving...")

    res = mail_agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "edit",
                        "edited_action" : {
                            "name" : "add_todo_tool",
                            "args" : {
                                "data" : "4 deployments pending for today"
                            }
                        }   
                    }
                ]
            }
        ),
        config2
    )

    print(res["messages"][-1].content)

📩 Approving...
I have added "4 deployments pending for today" in your todo list. Is there anything else?


In [57]:
todo

['2 deployments need to be done today', '4 deployments pending for today']